# 06 – Bronze: full load from the Dog CEO API into an Iceberg table + schema evolution

**Goal (Task 3, parts 2 and 3):**

1. Upgrade the Task 2 **full** API load (`replace`) so it writes an **Iceberg table** `bronze.dogceo_breeds`, partitioned by ingestion date.
2. Read it with **PyIceberg** and verify.
3. **Schema evolution:** add a new column, rerun, and show that the schema changed but the **existing data files were not rewritten**.

In [1]:
import os
import dlt
import pandas as pd
from dlt.sources.helpers import requests
from lake import PIPELINES_DIR, add_metadata
from pile import (BRONZE, get_catalog, pile_destination, partition_by_ingestion_date, run_to_pile,
                  list_pile, show_catalog_db, snapshots, data_files)

DOG_API_URL = os.getenv("DOG_API_URL", "https://dog.ceo/api/breeds/list/all")
pd.set_option("display.max_colwidth", 120)
catalog = get_catalog("silo")

## Step 1 – The resource

Same as notebook 03, with one switch: `new_column=...` adds one **extra column** to every row (used in step 4 for schema evolution).

`NEW_COLUMNS` has two choices. A column can be added only once, so if you already showed `is_sub_breed`, use `name_length` the next time.

In [2]:
NEW_COLUMNS = {
    "is_sub_breed": lambda breed, sub: sub is not None,                   # True for "australian shepherd"
    "name_length":  lambda breed, sub: len(f"{breed} {sub}" if sub else breed),
}


def build_api_resource(new_column: str = None):

    @dlt.resource(name="dogceo_breeds")
    def dog_breeds():
        response = requests.get(DOG_API_URL, timeout=30)
        response.raise_for_status()
        for breed, sub_breeds in response.json()["message"].items():
            for sub in (sub_breeds or [None]):          # [None] = breed without sub-breeds
                row = {
                    "breed": breed,
                    "sub_breed": sub,
                    "full_name": f"{breed} {sub}" if sub else breed,
                    "images_api": f"https://dog.ceo/api/breed/{breed}/{sub}/images" if sub
                                  else f"https://dog.ceo/api/breed/{breed}/images",
                }
                if new_column:                         # <- the NEW column (schema evolution)
                    row[new_column] = NEW_COLUMNS[new_column](breed, sub)
                yield row

    resource = dog_breeds()
    add_metadata(resource, source_system="dogceo", source_resource="breeds/list/all", ingestion_type="full")
    return partition_by_ingestion_date(resource)


api_pipeline = dlt.pipeline(
    pipeline_name="pile_api",
    destination=pile_destination("silo"),
    dataset_name=BRONZE,
    pipelines_dir=PIPELINES_DIR,
)

## Step 2 – Full load (`replace`) into Iceberg

In [3]:
run_to_pile(api_pipeline, build_api_resource(), write_disposition="replace")
print("Tables in bronze:", catalog.list_tables(BRONZE))
list_pile("bronze/dogceo_breeds")

/opt/conda/lib/python3.13/site-packages/pyiceberg/io/__init__.py:380: UserWarning: No preferred file implementation for scheme: memory
  if file_io := _infer_file_io_from_scheme(location, properties):


[silo] new rows loaded: {'dogceo_breeds': 165}
Tables in bronze: [('bronze', 'dogceo_breeds'), ('bronze', 'postgres_adopters'), ('bronze', 'postgres_animals'), ('bronze', 'postgres_applications'), ('bronze', 'postgres_shelters')]


,key,size_kb,last_modified
0,bronze/dogceo_breeds/data/ingestion_date=2026-11-06/00000-0-dfacfd1d-061d-4224-8083-715e5fe23f1b.parquet,10.7,2026-11-06 10:59:24.722+00:00
1,bronze/dogceo_breeds/metadata/00000-c75ef92e-ca2e-454b-aa75-d8a5088e0482.metadata.json,1.3,2026-11-06 10:59:24.530+00:00
2,bronze/dogceo_breeds/metadata/00001-6ead3c93-f82e-4410-80bd-a17c7e4df9ab.metadata.json,2.1,2026-11-06 10:59:24.879+00:00
3,bronze/dogceo_breeds/metadata/dfacfd1d-061d-4224-8083-715e5fe23f1b-m0.avro,5.2,2026-11-06 10:59:24.807+00:00
4,bronze/dogceo_breeds/metadata/snap-6779539742632945986-0-dfacfd1d-061d-4224-8083-715e5fe23f1b.avro,1.8,2026-11-06 10:59:24.840+00:00


## Step 3 – Read with PyIceberg and verify

Compare with a direct call to the API: same number of breeds, no duplicates, metadata filled.

In [4]:
breeds_table = catalog.load_table(f"{BRONZE}.dogceo_breeds")
breeds = breeds_table.scan().to_pandas()

api_data = requests.get(DOG_API_URL, timeout=30).json()["message"]
expected = sum(max(len(subs), 1) for subs in api_data.values())

print("Rows in Iceberg table:", len(breeds))
print("Rows expected from API:", expected, "→ OK" if len(breeds) == expected else "→ MISMATCH")
print("Duplicate full_name:  ", breeds.full_name.duplicated().sum())
print("Partition:", breeds_table.spec())
breeds.head()

Rows in Iceberg table: 165
Rows expected from API: 165 → OK
Duplicate full_name:   0
Partition: [
  1000: ingestion_date: day(7)
]


,breed,full_name,images_api,_source_system,_source_resource,_ingestion_type,_ingested_at,_load_id,_dlt_load_id,_dlt_id,sub_breed
0,affenpinscher,affenpinscher,https://dog.ceo/api/breed/affenpinscher/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,9Dwob85EA0tYDQ,NaN
1,african,african wild,https://dog.ceo/api/breed/african/wild/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,fdWgrKzbkiHu+w,wild
2,airedale,airedale,https://dog.ceo/api/breed/airedale/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,pV54jHz10ifw9Q,NaN
3,akita,akita,https://dog.ceo/api/breed/akita/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,9Ewqcu5Rf5XNBw,NaN
4,appenzeller,appenzeller,https://dog.ceo/api/breed/appenzeller/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,VQ+LxsIZMHNQOA,NaN


## Step 4 – Schema evolution

Save the **schema** and the **data files** BEFORE the change:

In [5]:
breeds_table = catalog.load_table(f"{BRONZE}.dogceo_breeds")
schema_before = breeds_table.schema()
snapshot_before = breeds_table.current_snapshot().snapshot_id
files_before = data_files(breeds_table)
objects_before = list_pile("bronze/dogceo_breeds/data")

print("Schema id BEFORE:", schema_before.schema_id)
print(schema_before)
files_before

Schema id BEFORE: 0
table {
  1: breed: optional string
  2: full_name: optional string
  3: images_api: optional string
  4: _source_system: optional string
  5: _source_resource: optional string
  6: _ingestion_type: optional string
  7: _ingested_at: optional timestamptz
  8: _load_id: optional string
  9: _dlt_load_id: required string
  10: _dlt_id: required string
  11: sub_breed: optional string
}


,file_path,record_count,file_size_in_bytes
0,bronze/dogceo_breeds/data/ingestion_date=2026-11-06/00000-0-dfacfd1d-061d-4224-8083-715e5fe23f1b.parquet,165,10956


Now rerun the import **with the new column** `is_sub_breed` (or `name_length` if `is_sub_breed` already exists). dlt sees the new column and asks Iceberg to **add it to the schema** (`update_schema().union_by_name(...)`).

In [6]:
run_to_pile(api_pipeline, build_api_resource(new_column="is_sub_breed"), write_disposition="replace")

[silo] new rows loaded: {'dogceo_breeds': 165}


In [7]:
breeds_table = catalog.load_table(f"{BRONZE}.dogceo_breeds")
schema_after = breeds_table.schema()
print("Schema id AFTER:", schema_after.schema_id)
print(schema_after)

Schema id AFTER: 1
table {
  1: breed: optional string
  2: full_name: optional string
  3: images_api: optional string
  4: _source_system: optional string
  5: _source_resource: optional string
  6: _ingestion_type: optional string
  7: _ingested_at: optional timestamptz
  8: _load_id: optional string
  9: _dlt_load_id: required string
  10: _dlt_id: required string
  11: sub_breed: optional string
  12: is_sub_breed: optional boolean
}


In [8]:
# Compare the two schemas
before_cols = {f.name for f in schema_before.fields}
after_cols = {f.name for f in schema_after.fields}
print("Added columns:  ", after_cols - before_cols)
print("Removed columns:", before_cols - after_cols)
print("\nAll schema versions Iceberg remembers:")
for schema_id, schema in breeds_table.schemas().items():
    print(f"  schema {schema_id}: {len(schema.fields)} columns")

Added columns:   {'is_sub_breed'}
Removed columns: set()

All schema versions Iceberg remembers:
  schema 0: 11 columns
  schema 1: 12 columns


### The existing data files were NOT rewritten

- The old Parquet file is still in the bucket with the **same key, size and last-modified time**.
- The old snapshot still points to it and is read with the **old schema** (without the new column).
- Adding a column only wrote **new metadata**. The new rows went into a **new** data file.

In [9]:
objects_after = list_pile("bronze/dogceo_breeds/data")
compare = objects_before.merge(objects_after, on="key", how="outer", suffixes=("_before", "_after"), indicator=True)
compare["_merge"] = compare["_merge"].map({"both": "unchanged (still there)", "right_only": "NEW file",
                                          "left_only": "removed"})
compare

,key,size_kb_before,last_modified_before,size_kb_after,last_modified_after,_merge
0,bronze/dogceo_breeds/data/ingestion_date=2026-11-06/00000-0-2ee71bc5-66c3-4e11-98e0-6d3218e1bd69.parquet,NaN,NaT,11.0,2026-11-06 10:59:29.346+00:00,NEW file
1,bronze/dogceo_breeds/data/ingestion_date=2026-11-06/00000-0-dfacfd1d-061d-4224-8083-715e5fe23f1b.parquet,10.7,2026-11-06 10:59:24.722+00:00,10.7,2026-11-06 10:59:24.722+00:00,unchanged (still there)


In [10]:
snapshots(breeds_table)      # replace = 'delete' + 'append' snapshot; schema_id goes 0 → 1

,snapshot_id,committed_at,operation,added_rows,deleted_rows,total_rows,schema_id
0,6779539742632945986,2026-11-06 10:59:24.845+00:00,append,165,0,165,0
1,2796389295146086649,2026-11-06 10:59:29.292+00:00,delete,0,165,0,1
2,6418873838537857917,2026-11-06 10:59:29.447+00:00,append,165,0,165,1


In [11]:
print("Old snapshot → its data files:")
display(data_files(breeds_table, snapshot_before))
print("Old snapshot read (old schema):")
display(breeds_table.scan(snapshot_id=snapshot_before).to_pandas().head(3))
print("Current table (new schema):")
breeds_table.scan().to_pandas()[["full_name", "sub_breed", *sorted(after_cols - before_cols)]].head(5)

Old snapshot → its data files:


,file_path,record_count,file_size_in_bytes
0,bronze/dogceo_breeds/data/ingestion_date=2026-11-06/00000-0-dfacfd1d-061d-4224-8083-715e5fe23f1b.parquet,165,10956


Old snapshot read (old schema):


,breed,full_name,images_api,_source_system,_source_resource,_ingestion_type,_ingested_at,_load_id,_dlt_load_id,_dlt_id,sub_breed
0,affenpinscher,affenpinscher,https://dog.ceo/api/breed/affenpinscher/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,9Dwob85EA0tYDQ,NaN
1,african,african wild,https://dog.ceo/api/breed/african/wild/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,fdWgrKzbkiHu+w,wild
2,airedale,airedale,https://dog.ceo/api/breed/airedale/images,dogceo,breeds/list/all,full,2026-11-06 10:59:19.942833+00:00,1793962760.4508278,1793962760.4508278,pV54jHz10ifw9Q,NaN


Current table (new schema):


,full_name,sub_breed,is_sub_breed
0,affenpinscher,NaN,False
1,african wild,wild,True
2,airedale,NaN,False
3,akita,NaN,False
4,appenzeller,NaN,False
